# 10. RQ4: Remediation-Time Modeling

**RQ4:** Among companies disclosing a material weakness since 2020, do modern ML
models identify different remediation-time drivers than the traditional
statistical methods used in Mojtahedi and Zhou (2024), which analyzed pre-2018 data?

**Method:** Multiple linear regression on `remediation_days`, supplemented by a Cox
proportional hazards survival model (handles right-censoring for weaknesses not
yet remediated at data cutoff).

---
## ⚠️ Status: NOT YET RUNNABLE — scaffold only, not executed

This notebook requires `remediation_days`, `weakness_category`, `industry_sic`,
`disclosure_date`, and `remediation_confirmed_date` — none of which exist yet,
because the SEC EDGAR extraction (`02_extract_pcaob_sec_edgar.ipynb`, Part 2) has
not actually been run against the live API. There is currently **no real data at
all** for RQ4 — this is the largest fully-open item in the project alongside RQ1's
code-metric mining.

The original code is preserved below, unmodified, for reference and so it's ready
to run the moment real SEC EDGAR data exists.


In [1]:
!pip install -q pandas statsmodels lifelines || pip install -q pandas statsmodels lifelines --break-system-packages

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## Original scaffold code (will raise a file-not-found / column error if run today)

In [2]:
import pandas as pd
import statsmodels.formula.api as smf
from lifelines import CoxPHFitter


def run_regression(df: pd.DataFrame):
    model = smf.ols(
        "remediation_days ~ C(weakness_category) + C(industry_sic) + inspection_year",
        data=df
    ).fit()
    print("=== RQ4: Multiple Regression (remediation_days) ===")
    print(model.summary())
    return model


def run_survival_model(df: pd.DataFrame):
    """
    event_observed = 1 if remediation_confirmed_date is present (event occurred),
    0 if right-censored (still unremediated at data cutoff).
    """
    surv_df = df.copy()
    surv_df["event_observed"] = surv_df["remediation_confirmed_date"].notna().astype(int)
    surv_df["duration"] = surv_df["remediation_days"].fillna(
        (pd.Timestamp("2026-07-17") - pd.to_datetime(surv_df["disclosure_date"])).dt.days
    )

    cols = ["duration", "event_observed", "weakness_category", "industry_sic"]
    surv_df = surv_df.dropna(subset=cols)
    surv_df = pd.get_dummies(surv_df[cols], columns=["weakness_category", "industry_sic"],
                              drop_first=True)

    cph = CoxPHFitter()
    cph.fit(surv_df, duration_col="duration", event_col="event_observed")
    print("\n=== RQ4: Cox Proportional Hazards Model (time-to-remediation) ===")
    cph.print_summary()
    print(f"\nConcordance index (C-statistic): {cph.concordance_index_:.3f}")
    return cph


def compare_to_prior_literature(model):
    """Narrative comparison against Mojtahedi and Zhou (2024)."""
    print("\n=== Comparison to Mojtahedi & Zhou (2024) ===")
    print("Significant drivers in this 2020+ ML model:")
    print(model.pvalues[model.pvalues < 0.05])
    print("\nCompare the above against the weakness categories and industry")
    print("associations reported in Mojtahedi and Zhou (2024) to assess H4.")

In [3]:
# NOT EXECUTED -- ../data/cleaned/audit_disclosure_dataset.csv currently has no
# remediation_days / weakness_category / industry_sic / disclosure_date /
# remediation_confirmed_date columns, since SEC EDGAR data has not been pulled.
#
# df = pd.read_csv("../data/cleaned/audit_disclosure_dataset.csv")
# reg_model = run_regression(df)
# run_survival_model(df)
# compare_to_prior_literature(reg_model)

print("Scaffold only -- see markdown cell above for what is blocking this notebook.")

Scaffold only -- see markdown cell above for what is blocking this notebook.
